# Name That LLM: offline submission (stage-2 Gemma)

Predicts the model behind each test passage and writes `submission.parquet`. Everything runs with **internet off**.

**Recipe** (checked on held-out training prompts, with uneven per-model counts, in a local simulation): the Gemma 3 4B LoRA classifier from the large training run (scores counted twice), an n-gram SVC trained on the same pool rows, and logistic regressions on Qwen3-Embedding and BERT features. The test set has no prompt ids, so passages are grouped into prompts by clustering their text, and each group's average is subtracted from the n-gram scores and the embedding features. That step added about 0.10 in the simulation. The simulated score was about 0.63-0.64. The leaderboard number can differ.

**Inputs to add** (the notebook finds them by name and stops with a clear message if one is missing)
1. The **Name That LLM** competition data.
2. The output of the training notebook `notebook6fafdc1c20`: adapter, head, training rows, base model copy and the packages in `offline/`.
3. The output of the **stage_frozen_models** notebook: the Qwen and BERT weights in `frozen/`.

**How to run it**
1. Settings: GPU T4 x2, **Internet off**.
2. **Save Version > Save & Run All**. I expect 20-30 minutes, which is an estimate. A commit uses a 3-row stand-in `test.csv`, so the log will say the test groups were not formed. That is expected. The real test (about 4,000 passages) is only used when you submit.
3. Read the log before submitting:
   - the install and `unsloth` import check pass;
   - **Gemma agreement with the training run is at least 97%** (this proves the adapter loaded correctly);
   - the **validation** line shows the recipe's score on 40 held-out training prompts;
   - `wrote /kaggle/working/submission.parquet`.
4. Then **Submit** the saved version.

If a step fails the notebook stops instead of quietly submitting something weaker (`REQUIRE_GEMMA`).

**This version** uses the stage-2 Gemma adapter (the stage-1 adapter fine-tuned on all official rows, from the `ntl-stage2-official` notebook) and adds the official rows to the n-gram training. Validation is switched off because the stage-2 adapter has seen every official row; instead the notebook checks that its Gemma scores match the ones the stage-2 run saved for 64 official rows. Add three notebook outputs as inputs: `notebook6fafdc1c20`, `ntl-stage-frozen-models` and `ntl-stage2-official`.


In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path

VALIDATE = False       # off: the stage-2 adapter has seen every official row, so held-out scores would be meaningless
VAL_PROMPTS = 40
VARIANT = 'recipe'     # 'recipe' (default), 'gemma_only' or 'no_centering': the last two exist only to probe the real test score
REQUIRE_GEMMA = True   # stop with an error rather than submit without the Gemma scores
USE_ASSIGNMENT = False # assignment scored 0.517 against 0.540 without it on the real test, so it stays off
NGRAM_OFFICIAL_WEIGHT = 1.0   # official rows join the n-gram training (+0.017 on held-out prompts in a local test)
def find(pattern, what):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if not hits: raise SystemExit(f'Could not find {what} (looked for {pattern}). Inputs present: {sorted(os.listdir("/kaggle/input"))}')
    return hits[0]

COMP = next((p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv')), None)
assert COMP, 'Add the Name That LLM competition data as an input.'
BASE = str(Path(find('offline/base_gemma/config.json', 'the Gemma base model copy')).parent)
WHEELS = find('offline/wheels', 'the saved packages')
ADAPTER = str(Path(find('stage2/runs/final/adapter/adapter_model.safetensors', 'the stage-2 LoRA adapter (add the ntl-stage2-official output)')).parent)
HEAD = find('stage2/runs/final/head.pt', 'the stage-2 classifier head')
SAVED = find('stage2/runs/final/pred.npz', 'the stage-2 scores on 64 official rows')
POOL = find('run/train_pool.parquet', 'the pool training rows')
QWEN = str(Path(find('frozen/qwen_emb/config.json', 'the Qwen3-Embedding weights (run the staging notebook)')).parent)
BERT = str(Path(find('frozen/bert/config.json', 'the BERT weights (run the staging notebook)')).parent)
for name, value in dict(COMP=COMP, BASE=BASE, WHEELS=WHEELS, ADAPTER=ADAPTER, HEAD=HEAD, SAVED=SAVED, POOL=POOL, QWEN=QWEN, BERT=BERT).items(): print(f'{name:8s} {value}')


In [ ]:
%%writefile score_gemma.py
"""Score passages with the trained Gemma LoRA classifier, offline. Mirrors the scoring path of worker_unsloth.py step for step:
same model load, same LoRA layout, same decoder, same three pooled layers, same head. Writes log-probabilities to --out.

  python score_gemma.py --check                                   # only import the stack and print versions
  python score_gemma.py --base DIR --adapter DIR --head head.pt --texts texts.parquet --out out.npz
"""
import argparse, importlib.metadata, importlib.util, sys, time
from pathlib import Path


def _torchao_too_old():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return False
    from packaging.version import parse
    return parse(version) < parse('0.16.0')


if _torchao_too_old():   # peft refuses old torchao builds. Nothing here uses torchao, so hide it before peft looks.
    _find_spec = importlib.util.find_spec
    importlib.util.find_spec = lambda name, *args, **kw: None if name.split('.')[0] == 'torchao' else _find_spec(name, *args, **kw)

ap = argparse.ArgumentParser()
ap.add_argument('--check', action='store_true')
ap.add_argument('--base'); ap.add_argument('--adapter'); ap.add_argument('--head'); ap.add_argument('--texts'); ap.add_argument('--out')
ap.add_argument('--max-len', type=int, default=384); ap.add_argument('--seed', type=int, default=1); ap.add_argument('--rank', type=int, default=16)
a = ap.parse_args()

import unsloth  # noqa: F401  (must load before transformers and peft)
from unsloth import FastModel
import numpy as np, pandas as pd, torch
from torch import nn

if a.check:
    import bitsandbytes, peft, transformers
    print('unsloth', unsloth.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__,
          '| bitsandbytes', bitsandbytes.__version__, '| gpus', torch.cuda.device_count())
    sys.exit(0)

T0 = time.time()
def log(*x): print(f'[{(time.time() - T0) / 60:6.1f}m]', *x, flush=True)

dev = 'cuda'; compute = torch.float16    # the training run used the default, fp16 (Unsloth keeps Gemma's activations in fp32 itself)
texts = pd.read_parquet(a.texts).text.fillna('').astype(str).tolist()

model, processor = FastModel.from_pretrained(model_name=a.base, max_seq_length=a.max_len, dtype=compute, load_in_4bit=True,
                                             full_finetuning=False, use_gradient_checkpointing='unsloth', random_state=a.seed)
tok = getattr(processor, 'tokenizer', processor)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'right'
model = FastModel.get_peft_model(model, r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0, bias='none', random_state=a.seed,
                                 finetune_vision_layers=False, finetune_language_layers=True,
                                 finetune_attention_modules=True, finetune_mlp_modules=True, use_gradient_checkpointing='unsloth')
log('loaded base model and LoRA layout')


def find_decoder(m):
    best = None
    for name, mod in m.named_modules():
        if any(w in name.lower() for w in ('vision', 'audio', 'visual')): continue
        layers = getattr(mod, 'layers', None)
        if isinstance(layers, nn.ModuleList) and hasattr(mod, 'embed_tokens') and (best is None or len(layers) > len(best[1].layers)):
            best = (name, mod)
    if best is None: raise SystemExit('could not find the text decoder inside the model')
    return best
dec_name, decoder = find_decoder(model)
n_layers = len(decoder.layers); picks = sorted({n_layers // 2, 3 * n_layers // 4, n_layers})
hidden = int(getattr(getattr(model.config, 'text_config', model.config), 'hidden_size'))
log('decoder', dec_name or '(root)', n_layers, 'layers, pooling', picks)


class Head(nn.Module):
    def __init__(self):
        super().__init__(); self.norms = nn.ModuleList(nn.LayerNorm(hidden) for _ in picks); self.out = nn.Linear(hidden * len(picks), 18)
    def features(self, pooled): return torch.cat([n(p) for n, p in zip(self.norms, pooled)], -1)
    def forward(self, pooled): return self.out(self.features(pooled))
head = Head().to(dev)

# same order as training: for_training first, then adapters to fp32, so the saved fp32 weights load without rounding
FastModel.for_training(model)
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()

import json
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
cfg = json.loads((Path(a.adapter) / 'adapter_config.json').read_text())
if int(cfg.get('r', a.rank)) != a.rank: raise SystemExit(f'adapter rank {cfg.get("r")} differs from --rank {a.rank}')
state = load_file(str(Path(a.adapter) / 'adapter_model.safetensors'))
result = set_peft_model_state_dict(model, state)
unexpected = list(getattr(result, 'unexpected_keys', None) or [])
if unexpected: raise SystemExit(f'{len(unexpected)} adapter tensors did not match the model, first: {unexpected[:3]}')
lora_b = sum(float(p.detach().float().abs().sum()) for n, p in model.named_parameters() if 'lora_B' in n)
if not lora_b > 0: raise SystemExit('every LoRA B matrix is still zero after loading: the adapter was not applied')
head.load_state_dict(torch.load(a.head, map_location='cpu'))
log(f'adapter loaded ({len(state)} tensors, |B| sum {lora_b:.1f}) and head loaded')

enc = tok(texts, truncation=True, max_length=a.max_len)['input_ids']
def batch(rows): return tok.pad({'input_ids': [enc[int(i)] for i in rows]}, return_tensors='pt').to(dev)
def pool_layers(b):
    with torch.autocast(dev, dtype=compute):
        hs = decoder(input_ids=b['input_ids'], attention_mask=b['attention_mask'], output_hidden_states=True, use_cache=False).hidden_states
    if hs is None or len(hs) != n_layers + 1: raise SystemExit(f'the decoder returned {None if hs is None else len(hs)} hidden states, expected {n_layers + 1}')
    m = b['attention_mask'].clone().float(); m[:, 0] = 0; m = m.unsqueeze(-1); den = m.sum(1).clamp(min=1)
    return [(hs[l].float() * m).sum(1) / den for l in picks]

model.eval(); head.eval()
N = len(enc); logp = np.zeros((N, 18), np.float32)
with torch.inference_mode():
    idx = np.argsort([len(x) for x in enc], kind='stable')
    for s in range(0, N, 16):
        rows = idx[s:s + 16]
        logp[rows] = torch.log_softmax(head.out(head.features(pool_layers(batch(rows)))), -1).cpu().numpy()
        if (s // 16) % 50 == 0: log(f'scored {s}/{N}')
if not np.isfinite(logp).all(): raise SystemExit('non-finite predictions')
np.savez(a.out, logp=logp)
log(f'done: {N} passages scored, saved to {a.out}')


In [ ]:
%%writefile ntl_pipeline.py
"""Name That LLM: offline submission pipeline.

Recipe (validated in experiments/sim_unequal.py): 2 x Gemma LoRA scores (raw) + n-gram scores (centred per prompt group)
+ Qwen3-Embedding mid-layer LR (centred) + BERT mid-layer LR (centred). "Centred" means each prompt group's mean is subtracted.
The test set has no prompt ids, so groups are estimated from the text: Ward clustering of Qwen last-layer features, with the
number of clusters chosen by silhouette (on training rows it lands within 0.91-1.08x of the true prompt count).

run_pipeline() takes the Gemma scorer as a function so everything else can be tested without Unsloth.
"""
import json, time
from pathlib import Path
import numpy as np, pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.special import log_softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.svm import LinearSVC

WEIGHTS = {'gemma': 2.0, 'ngram': 1.0, 'qwen': 1.0, 'bert': 1.0}


def macro_recall(y, p):
    return float(np.mean([(p[y == c] == c).mean() for c in np.unique(y)]))


def center_by_group(X, groups):
    X = np.asarray(X, dtype=np.float64).copy()
    for g in np.unique(groups):
        m = groups == g
        X[m] -= X[m].mean(0, keepdims=True)
    return X


def assign_by_group(S, groups):
    """One model per passage inside each group, each model used at most ceil(size/18) times. Only valid if prompts really
    hold one passage per model, which the competition says is not guaranteed. Off by default."""
    pred = S.argmax(1).copy(); k = S.shape[1]
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        r, c = linear_sum_assignment(-np.tile(S[idx], (1, int(np.ceil(len(idx) / k)))))
        pred[idx[r]] = c % k
    return pred


def estimate_groups(X, lo=40, hi=4, n_grid=24):
    """Cluster passages into prompt groups from text features alone. Returns (labels, info)."""
    N = len(X)
    if N < 30:
        return np.zeros(N, dtype=int), {'K': 1, 'note': f'only {N} rows: no grouping'}
    Xn = normalize(X - X.mean(0))
    Z = linkage(Xn, 'ward'); D = pairwise_distances(Xn)
    best = None
    for K in sorted({int(round(k)) for k in np.geomspace(max(2, N / lo), max(3, N / hi), n_grid)}):
        lab = fcluster(Z, K, 'maxclust')
        if not 1 < len(np.unique(lab)) < N: continue
        s = silhouette_score(D, lab, metric='precomputed')
        if best is None or s > best[1]: best = (K, s, lab)
    if best is None:
        return np.zeros(N, dtype=int), {'K': 1, 'note': 'no usable clustering'}
    K, s, lab = best
    lab = np.unique(lab, return_inverse=True)[1]; sizes = np.bincount(lab)
    return lab, {'K': int(len(sizes)), 'silhouette': float(s), 'size_min': int(sizes.min()), 'size_median': float(np.median(sizes)),
                 'size_mean': float(sizes.mean()), 'size_max': int(sizes.max()), 'singletons': int((sizes == 1).sum())}


def ngram_scores(pool, labels, text_sets, log=print, extra=None, extra_weight=1.0):
    """The same n-gram SVC the training notebook scored the official rows with, trained on the pool rows (plus, optionally, official rows
    at extra_weight: extra = (texts, labels))."""
    t = time.time()
    svc = Pipeline([('f', FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=300000, sublinear_tf=True, lowercase=False, dtype=np.float32)),
        ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=100000, sublinear_tf=True, lowercase=False, token_pattern=r'\w+|[^\w\s]', dtype=np.float32))])),
        ('clf', LinearSVC(C=0.5, class_weight='balanced', max_iter=5000))])
    X, Y = pool.text.tolist(), pool.model.tolist(); sw = np.ones(len(X))
    if extra is not None:
        X += list(extra[0]); Y += list(extra[1]); sw = np.r_[sw, np.full(len(extra[0]), float(extra_weight))]
    svc.fit(X, Y, clf__sample_weight=sw)
    cols = [list(svc.classes_).index(l) for l in labels]
    log(f'n-gram fitted on {len(X)} rows ({len(pool)} pool + {len(X) - len(pool)} official) in {(time.time() - t) / 60:.1f} min')
    return [log_softmax(svc.decision_function(texts)[:, cols] * 4, axis=1) if len(texts) else np.zeros((0, len(labels))) for texts in text_sets]


def embed(path, texts, depths, log=print, batch=16, max_len=512):
    """Mean-pooled hidden states. depths are fractions of the layer count (0.5 = middle, 1.0 = last). Returns {layer: array}."""
    import torch
    from transformers import AutoModel, AutoTokenizer
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModel.from_pretrained(path).float().to(dev).eval()
    n = model.config.num_hidden_layers; layers = sorted({max(1, round(f * n)) for f in depths})
    texts = list(texts); order = np.argsort([len(t) for t in texts]); out = {L: np.zeros((len(texts), model.config.hidden_size), np.float32) for L in layers}
    t = time.time()
    with torch.no_grad():
        for i in range(0, len(order), batch):
            idx = order[i:i + batch]
            enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(dev)
            hs = model(**enc, output_hidden_states=True).hidden_states
            m = enc['attention_mask'].unsqueeze(-1).float()
            for L in layers: out[L][idx] = ((hs[L].float() * m).sum(1) / m.sum(1)).cpu().numpy()
    del model
    if dev == 'cuda': torch.cuda.empty_cache()
    log(f'embedded {len(texts)} passages with {Path(path).name} ({n} layers, pooled {layers}) in {(time.time() - t) / 60:.1f} min')
    return out


def frozen_scores(E_train, y_train, g_train, E_apply, g_apply, C=0.01, center=True):
    """Logistic regression on prompt-centred features. Training rows are centred by their true prompt, rows to score by their estimated group."""
    Xtr = center_by_group(E_train, g_train) if center else np.asarray(E_train, dtype=np.float64); sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(C=C, max_iter=400).fit(sc.transform(Xtr), y_train)
    return clf.decision_function(sc.transform(center_by_group(E_apply, g_apply) if center else np.asarray(E_apply, dtype=np.float64)))


def blend(parts, weights=WEIGHTS):
    return sum(weights[k] * (v / v.std()) for k, v in parts.items())


def run_pipeline(comp_dir, pool_path, qwen_dir, bert_dir, gemma_fn, out_dir, validate=True, val_prompts=40, require_gemma=True,
                 use_assignment=False, holdout_prompts=(), test_df=None, pool_limit=None, variant='recipe', ngram_official_weight=0.0, log=print):
    """variant: 'recipe' (default), 'gemma_only' (Gemma scores alone, to see what it really scores on the test) or 'no_centering'."""
    t0 = time.time(); out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(f'{comp_dir}/train.csv'); train['text'] = train.text.fillna('').astype(str)
    test = test_df.copy() if test_df is not None else pd.read_csv(f'{comp_dir}/test.csv'); test['text'] = test.text.fillna('').astype(str)
    labels = sorted(train.model.unique()); y = train.model.map({l: i for i, l in enumerate(labels)}).to_numpy(); g_true = train.prompt_id.to_numpy()
    usable = ~np.isin(g_true, list(holdout_prompts))   # rows allowed to train the frozen models (everything, except in local tests)
    rng = np.random.default_rng(0)
    val_set = set(rng.choice(np.unique(g_true[usable]), val_prompts, replace=False)) if validate else set()
    val_idx = np.where(np.isin(g_true, list(val_set)) & usable)[0]
    log(f'{len(train)} train rows, {len(test)} test rows; validation on {len(val_idx)} held-out train rows ({len(val_set)} prompts)')

    # 1. Gemma LoRA scores for the validation rows and the test rows in one call
    texts = list(train.text.iloc[val_idx]) + list(test.text)
    try:
        G = np.asarray(gemma_fn(texts), dtype=np.float64); gemma_ok = True
    except Exception as error:
        if require_gemma: raise
        log(f'!!! GEMMA FAILED, continuing without it: {error!r}'); G = np.zeros((len(texts), len(labels))); gemma_ok = False
    G_val, G_test = G[:len(val_idx)], G[len(val_idx):]

    # 2. n-gram scores from the pool rows
    pool = pd.read_parquet(pool_path, columns=['text', 'model'])
    if pool_limit: pool = pool.sample(min(pool_limit, len(pool)), random_state=0)
    extra = None
    if ngram_official_weight > 0:   # official rows join the n-gram training, except validation rows so validation stays honest
        keep = usable.copy(); keep[val_idx] = False
        extra = (list(train.text[keep]), list(train.model[keep]))
    N_val, N_test = ngram_scores(pool, labels, [train.text.iloc[val_idx], test.text], log, extra=extra, extra_weight=ngram_official_weight)

    # 3. frozen embedders on all training rows and on the test rows
    q_tr = embed(qwen_dir, train.text, (0.5, 1.0), log); q_te = embed(qwen_dir, test.text, (0.5, 1.0), log)
    b_tr = embed(bert_dir, train.text, (0.5,), log); b_te = embed(bert_dir, test.text, (0.5,), log)
    qmid, qlast = sorted(q_tr)[0], sorted(q_tr)[-1]; bmid = sorted(b_tr)[0]

    def recipe(G_, N_, qs_tr_mask, q_apply, b_apply, last_apply, true_labels=None):
        est, info = estimate_groups(last_apply)
        if variant == 'gemma_only':
            parts = {'gemma': G_}
        else:
            c = variant != 'no_centering'
            fq = frozen_scores(q_tr[qmid][qs_tr_mask], y[qs_tr_mask], g_true[qs_tr_mask], q_apply, est, center=c)
            fb = frozen_scores(b_tr[bmid][qs_tr_mask], y[qs_tr_mask], g_true[qs_tr_mask], b_apply, est, center=c)
            parts = {'gemma': G_, 'ngram': center_by_group(N_, est) if c else N_, 'qwen': fq, 'bert': fb}
        if not gemma_ok: parts.pop('gemma', None)
        S = blend(parts, {k: WEIGHTS[k] for k in parts})
        pred = assign_by_group(S, est) if use_assignment else S.argmax(1)
        return S, pred, parts, est, info

    report = {'weights': WEIGHTS, 'gemma_used': gemma_ok, 'use_assignment': bool(use_assignment), 'variant': variant, 'ngram_official_weight': ngram_official_weight}
    # 4. validation: frozen models see only training prompts outside the validation prompts
    if validate and len(val_idx):
        mask = usable.copy(); mask[val_idx] = False
        S, pred, parts, est, info = recipe(G_val, N_val, mask, q_tr[qmid][val_idx], b_tr[bmid][val_idx], q_tr[qlast][val_idx])
        yv = y[val_idx]
        report['validation'] = {'rows': int(len(val_idx)), 'groups': info, 'recipe': macro_recall(yv, pred),
                                'components': {k: macro_recall(yv, v.argmax(1)) for k, v in parts.items()}}
        log('validation (held-out training prompts):', json.dumps(report['validation']))
        report['_val_logp'] = (val_idx, G_val)
    # 5. test: frozen models see every usable training row
    S, pred, parts, est, info = recipe(G_test, N_test, usable, q_te[qmid], b_te[bmid], q_te[qlast])
    report['test_groups'] = info; log('test groups:', json.dumps(info))
    sub = pd.DataFrame({'id': test['id'].astype(str), 'model': [labels[i] for i in pred]})
    assert sub.id.is_unique and sub.model.notna().all() and set(sub.model) <= set(labels), 'invalid submission'
    sub.to_parquet(out_dir / 'submission.parquet', index=False)
    report['test_prediction_counts'] = sub.model.value_counts().to_dict(); report['minutes'] = (time.time() - t0) / 60
    (out_dir / 'submission_info.json').write_text(json.dumps({k: v for k, v in report.items() if not k.startswith('_')}, indent=1))
    log(f'wrote {out_dir / "submission.parquet"} ({len(sub)} rows) in {report["minutes"]:.1f} min')
    report['_pred'] = pred; report['_labels'] = labels
    return report


## Install the packages offline and check the stack imports


In [ ]:
wheels = sorted(Path(WHEELS).iterdir())
print(f'installing {len(wheels)} packages: ' + ', '.join(w.name for w in wheels))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q', *map(str, wheels)], capture_output=True, text=True)
print((r.stdout + r.stderr)[-3000:])
if r.returncode: raise SystemExit('The offline install failed: read the pip output above.')
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
check = subprocess.run([sys.executable, 'score_gemma.py', '--check'], capture_output=True, text=True, env=env)
print('\n'.join(l for l in check.stdout.splitlines() if not l.startswith(('🦥', '==((', '   \\', 'O^O', '\\ ', ' "-'))))
if check.returncode: raise SystemExit('Unsloth did not import after the offline install:\n' + check.stderr[-3000:])


## Run the pipeline and write submission.parquet


In [ ]:
import numpy as np, pandas as pd
sys.path.insert(0, '.')
import ntl_pipeline as P

def gemma_fn(texts):
    pd.DataFrame({'text': texts}).to_parquet('/kaggle/working/_texts.parquet')
    r = subprocess.run([sys.executable, 'score_gemma.py', '--base', BASE, '--adapter', ADAPTER, '--head', HEAD, '--texts', '/kaggle/working/_texts.parquet',
                        '--out', '/kaggle/working/_gemma.npz'], capture_output=True, text=True, env=env)
    print('\n'.join(l for l in r.stdout.splitlines() if l.startswith('[')))
    if r.returncode: raise RuntimeError('Gemma scoring failed:\n' + (r.stderr or r.stdout)[-3000:])
    return np.load('/kaggle/working/_gemma.npz')['logp']

report = P.run_pipeline(COMP, POOL, QWEN, BERT, gemma_fn, '/kaggle/working', validate=VALIDATE, val_prompts=VAL_PROMPTS,
                        require_gemma=REQUIRE_GEMMA, use_assignment=USE_ASSIGNMENT, variant=VARIANT, ngram_official_weight=NGRAM_OFFICIAL_WEIGHT)

# proof that the stage-2 adapter and head loaded exactly as trained: score the 64 official rows the stage-2 run saved scores for
chk = pd.read_csv(f'{COMP}/train.csv').text.fillna('').astype(str).head(64).tolist()
got = gemma_fn(chk); saved = np.load(SAVED)['logp']
same = float((got.argmax(1) == saved.argmax(1)).mean())
print(f'Gemma agreement with the stage-2 run on {len(chk)} rows: {same:.1%} (max score difference {np.abs(got - saved).max():.3f})')
if same < 0.95 and REQUIRE_GEMMA:
    raise SystemExit('The Gemma scores do not match the stage-2 run, so the adapter or model setup differs. Do not submit this version.')


## Result


In [ ]:
print(Path('/kaggle/working/submission_info.json').read_text())
sub = pd.read_parquet('/kaggle/working/submission.parquet'); print(sub.shape, sub.dtypes.to_dict()); print(sub.head())
